In [ ]:
import sys,os,re
import numpy             as np
import matplotlib.pyplot as plt
import pandas            as pd
import seaborn           as sb

from source_code.galdist import galaxy_distribution

from itertools import product
from copy      import deepcopy
from time      import time

from scipy.interpolate import interp1d
from scipy.integrate   import trapz

import warnings
warnings.filterwarnings('ignore')

import matplotlib
from matplotlib import rc
from matplotlib.pyplot import cm
from matplotlib.colors import LogNorm

rc('text', usetex=True)
rc('font', family='serif')
matplotlib.rcParams.update({'font.size': 18})

sidelegend = {'bbox_to_anchor': (1.04,0.5), 
              'loc': "center left",
              'frameon': False}


In [ ]:
#Euclid survey specifications, N_gw in [10^5-10^6] according to ET
galaxy_specs = {'fsky': 0.35, 
                'gal_per_arcmin': 30.,
                'sigma_eps': 0.3,  #sigma associated to noise for GC and WL
                'Nbin_ell': 20,
                'lmin': 10,
                'lmax': 1500}

GW_specs = {'fsky': 0.35, 
            'N_gw': 10**5, 
            'sigma_eps_gw': 0.005} #sigma associated to noise (d_L) for GW-WL

analysis_settings = {'Nbin_ell': 20,
                     'lmin': 10,
                     'lmax': 1500}

use_obs  = ['GC','WL','GWWL', 'GWC']


In [ ]:
lmin = np.log10(analysis_settings['lmin'])
lmax = np.log10(analysis_settings['lmax'])
N    = analysis_settings['Nbin_ell']

ell_lims = np.logspace(lmin,lmax,N) #creation of array-> N bin log spaced
ells     = np.array([int(ell) for ell in 0.5*(ell_lims[:-1]+ell_lims[1:])]) 
#evaluation of middle points of each bin 
deltas   = (ell_lims[1:]-ell_lims[:-1]) #evaluation of the amplitude of each bin

In [ ]:
fiducial = {'ombh2': 0.022445,
            'omch2': 0.1205579307,
            'ns': 0.96,
            'As': 2.12605e-09,
            'tau': 0.05,
            'H0': 67.,
            'w': -1.,
            'wa': 0.,
            'mnu': 0.06,
            'a0': - 0.007589,
            'a1' :  0.002008,
            'a2' : - 0.004127,
            'a3' :  0.002918,
            'a4' : -0.0006784,
            #'A_IA': 1.72,
            #'eta_IA': -0.41,
            'b0_poly': 0.830703,
            'b1_poly': 1.190547,
            'b2_poly': -0.928357,
            'b3_poly': 0.423292}

In [ ]:
distributions = np.load('./mock_data/LCDM_test_galGWs_source_distribution.npy',allow_pickle=True).item()


In [ ]:
maxbins=0
if 'GC' in use_obs:
    Nbins_gc=distributions['GC']['Nbins']
    maxbins=max(Nbins_gc, maxbins)
if 'WL' in use_obs:
    Nbins_wl=distributions['WL']['Nbins']
    maxbins=max(Nbins_wl, maxbins)
if 'GWWL' in use_obs:
    Nbins_gwl=distributions['GWWL']['Nbins']
    maxbins=max(Nbins_gwl, maxbins)
if 'GWC' in use_obs:
    Nbins_gwc=distributions['GWC']['Nbins']
    maxbins=max(Nbins_gwc, maxbins)

In [ ]:
if 'WL' in use_obs: 
    WLcols  = ['L{}xL{}'.format(i,j) for i in range(1,Nbins_wl+1) for j in range(i,Nbins_wl+1)]
if 'GC' in use_obs: 
    GCcols  = ['G{}xG{}'.format(i,j) for i in range(1,Nbins_gc+1) for j in range(i,Nbins_gc+1)]    
if 'GWWL' in use_obs: 
    GWWLcols  = ['WL{}xWL{}'.format(i,j) for i in range(1,Nbins_gwl+1) for j in range(i,Nbins_gwl+1)]
if 'GWC' in use_obs: 
    GWCcols  = ['WC{}xWC{}'.format(i,j) for i in range(1,Nbins_gwc+1) for j in range(i,Nbins_gwc+1)]
if 'GC' in  use_obs and 'WL' in use_obs: 
    GGLcols = ['G{}xL{}'.format(i,j) for i in range(1,Nbins_gc+1) for j in range(1,Nbins_wl+1)]
if 'WL' in use_obs and 'GWC' in use_obs:
    LGWCcols = ['L{}xWC{}'.format(i,j) for i in range(1,Nbins_wl+1) for j in range(1,Nbins_gwc+1)]
if 'GC' in  use_obs and 'GWC' in  use_obs:
    GGWCcols = ['G{}xWC{}'.format(i,j) for i in range(1,Nbins_gc+1) for j in range(1,Nbins_gwc+1)]
if 'GC' in  use_obs and 'GWWL' in  use_obs:
    GGWLcols = ['G{}xWL{}'.format(i,j) for i in range(1,Nbins_gc+1) for j in range(1,Nbins_gwl+1)]
if 'WL' in use_obs and 'GWWL' in use_obs:
    LGWLcols = ['L{}xWL{}'.format(i,j) for i in range(1,Nbins_wl+1) for j in range(1,Nbins_gwl+1)]
if 'GWC' in use_obs and 'GWWL' in use_obs:
    GWCGWLcols = ['WC{}xWL{}'.format(i,j) for i in range(1,Nbins_gwc+1) for j in range(1,Nbins_gwl+1)]

In [ ]:
all_cols=[]
if 'WL' in use_obs: 
    all_cols = all_cols+WLcols
if 'GC' in use_obs:
    if 'WL' in use_obs:
        all_cols = all_cols+GGLcols+GCcols
    else:
        all_cols = all_cols+GCcols
if 'GWC' in use_obs:
    all_cols = all_cols+GWCcols
    if 'GC' in use_obs:
        all_cols = all_cols+GGWCcols
    if 'WL' in use_obs:
        all_cols = all_cols+LGWCcols
if 'GWWL' in use_obs:
    all_cols = all_cols + GWWLcols
    if 'GC' in use_obs:
        all_cols = all_cols+GGWLcols
    if 'WL' in use_obs:
        all_cols = all_cols+LGWLcols
    if 'GWC' in use_obs:
        all_cols = all_cols+GWCGWLcols


In [ ]:

covmat_dict = np.load('./mock_data/LCDM_test_galGWs_covmat.npy',allow_pickle=True).item()
Cls= pd.read_csv('./mock_data/LCDM_test_galGWs_Cls_noiseless.dat',sep='\s+',header=0)
sigma = pd.DataFrame(columns=['ells']+all_cols)

sigma['ells'] = [int(ell) for ell in ells]

for col in all_cols:
    sigma[col] = 0
    
for ind, ell in enumerate(ells): # Access the covariance matrix
    for col in all_cols:
        sigma.at[ind, col] = np.sqrt(covmat_dict[str(int(ell))].at[col,col])


In [ ]:
plot_path = './plot/Validation_test_LCDM_'

    
if 'GC' in use_obs and 'WL' in use_obs:
    plot_path = plot_path+'gal'
elif 'GC' in use_obs:
    plot_path = plot_path+'GC'
elif 'WL' in use_obs:
    plot_path = plot_path+'WL'
if 'GWWL' in use_obs and 'GWC' in use_obs:
        plot_path = plot_path+'GWs'
elif 'GWC' in use_obs:
        plot_path = plot_path+'GWC'
elif 'GWWL' in use_obs:
        plot_path = plot_path+'GWWL'

import camb

In [ ]:
from source_code.compute_obs_sources import get_obs
extra={}

In [ ]:
MG_params={'MG_flag': 0}
fiducial.update(MG_params)
settings={'camb_path':camb.__path__,#
         'case': 'simple',
         'calculation': 'CAMB',
          'extra':extra}

In [ ]:
calc_obs_MGCAMB =  get_obs(fiducial,distributions,ells,settings,feedback=True) #pd.read_csv('./Validation_data/Cls_MGCAMB.dat', sep='\t')


In [ ]:
#calc_obs_CAMB.Cls.to_csv('./Validation_data/Cls_CAMB.dat',  sep='\t', index=False, header=True)


In [ ]:
camb_submodules = ['camb.baseconfig', 'camb.reionization', 'camb.recombination', 'camb.constants', 
        'camb.initialpower', 'camb.nonlinear', 'camb.dark_energy', 'camb.sources', 
        'camb.bbn', 'camb.model', 'camb._config', 'camb.results', 'camb.camb', 'camb.mathutils']

for module in camb_submodules:
    if module in sys.modules:
        del sys.modules[module]
    
if 'camb' in sys.modules:
    del sys.modules['camb']

del fiducial['MG_flag']

In [ ]:
settings['camb_path']='/Users/chiaradeleo/Desktop/camb_fork/GW-CAMB'

calc_obs_CAMB = get_obs(fiducial,distributions,ells,settings,feedback=True)
#calc_obs_CAMB = pd.read_csv('./Validation_data/Cls_CAMB.dat', sep='\t')

In [ ]:
#calc_obs_MGCAMB.Cls.to_csv('./Validation_data/Cls_MGCAMB.dat',  sep='\t', index=False, header=True)
calc_obs_MGCAMB = calc_obs_MGCAMB.Cls
calc_obs_CAMB = calc_obs_CAMB.Cls

In [ ]:
if len(use_obs)==1:
    Ncols=1
elif len(use_obs)==2:
    Ncols=3
elif len(use_obs)==3:
    Ncols=6
else:
    Ncols=10
bincolors = sb.color_palette('rainbow',maxbins)



In [ ]:
norm = ells*(ells+1)/(2*np.pi)
j=0
if 'GWWL' in use_obs or 'GWC'  in use_obs or 'GC' in use_obs:
    fig, axes = plt.subplots(ncols=len(use_obs), sharey=True, subplot_kw=dict(frameon=True),figsize=(12,4))
    plt.suptitle('Difference between $C_\ell$ evaluated with CAMB and MGCAMB',y=1.05)
    axis_label_fontsize = 20
    title_fontsize = 20
    title_fontweight = 'bold'
    if 'GC' in use_obs:
        axes[j].set_title('GCph',loc='right',fontdict={'fontsize':16})
        for i in range(1,Nbins_gc+1):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['G{}xG{}'.format(i,i)]-calc_obs_CAMB['G{}xG{}'.format(i,i)]),#/calc_obs_MGCAMB['G{}xG{}'.format(i,i)],
                         label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1
        
    if 'WL' in use_obs:
        axes[j].set_title('WL',loc='right',fontdict={'fontsize':16})
        for i in range(1,Nbins_wl+1):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['L{}xL{}'.format(i,i)]-calc_obs_CAMB['L{}xL{}'.format(i,i)]),#/calc_obs_MGCAMB['L{}xL{}'.format(i,i)],
                         label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[j].set_xlabel(r'$\ell$')
        #axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1    
    
    if 'GWWL' in use_obs:
        axes[j].set_title('GW-WL',loc='right',fontdict={'fontsize':16})
        for i in range(1,Nbins_gwl+1):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['WL{}xWL{}'.format(i,i)]-calc_obs_CAMB['WL{}xWL{}'.format(i,i)]),#,/calc_obs_MGCAMB['WL{}xWL{}'.format(i,i)],
                         label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[j].set_xlabel(r'$\ell$')
        #axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1

    if 'GWC' in use_obs:
        axes[j].set_title('GW-NC',loc='right',fontdict={'fontsize':16})
        for i in range(1,Nbins_gwc+1):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['WC{}xWC{}'.format(i,i)]-calc_obs_CAMB['WC{}xWC{}'.format(i,i)]),#/calc_obs_MGCAMB['WC{}xWC{}'.format(i,i)],
                         label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[j].set_xlabel(r'$\ell$')
        #axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1
        
    
    
    
    axes[-1].legend(**sidelegend)
    axes[0].set_ylabel(r'$\Delta C_{ii}^{AB}(\ell)$', fontsize=axis_label_fontsize)
    plt.subplots_adjust(hspace=1.2)
    #plt.savefig(plot_path+'.pdf', bbox_inches='tight')
    axes[-1].legend(**sidelegend);
    

In [ ]:
norm = ells*(ells+1)/(2*np.pi)
j=0
if 'GWWL' in use_obs or 'GWC'  in use_obs or 'GC' in use_obs:
    fig, axes = plt.subplots(ncols=len(use_obs)+2, sharey=False, subplot_kw=dict(frameon=True),figsize=(16,6))
    plt.suptitle('Relative difference between $C_\ell$ evaluated with CAMB and MGCAMB',y=1.05)
    axis_label_fontsize = 20
    title_fontsize = 20
    title_fontweight = 'bold'
    

    
    
    if 'GC' in use_obs and 'WL' in use_obs:
        axes[j].set_title('GGL',loc='right',fontdict={'fontsize':16})
        for i in range(1,Nbins_gc+1):
            axes[j].plot(ells,(calc_obs_MGCAMB['G{}xL{}'.format(i,i)]-calc_obs_CAMB['G{}xL{}'.format(i,i)])/calc_obs_MGCAMB['G{}xL{}'.format(i,i)],
                        label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
        
    if 'GC' in use_obs and 'GWWL' in use_obs:
        axes[j].set_title('GCphxGW-WL',loc='right',fontdict={'fontsize':16})
        for i in range(1,Nbins_gc+1):
            axes[j].plot(ells,(calc_obs_MGCAMB['G{}xWL{}'.format(i,i)]-calc_obs_CAMB['G{}xWL{}'.format(i,i)])/calc_obs_MGCAMB['G{}xWL{}'.format(i,i)],
                        label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
       
    
    
    if 'WL' in use_obs and 'GWWL' in use_obs:
        axes[j].set_title('WLxGW-WL',loc='right',fontdict={'fontsize':16})
        for i in range(1,Nbins_gc+1):
            axes[j].plot(ells,(calc_obs_MGCAMB['L{}xWL{}'.format(i,i)]-calc_obs_CAMB['L{}xWL{}'.format(i,i)])/calc_obs_MGCAMB['L{}xWL{}'.format(i,i)],
                        label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
        
    if 'GC' in use_obs and 'GWC' in use_obs:
        axes[j].set_title('GCphxGW-NC',loc='right',fontdict={'fontsize':16})
        for i in range(1,Nbins_gc+1):
            axes[j].plot(ells,(calc_obs_MGCAMB['G{}xWC{}'.format(i,i)]-calc_obs_CAMB['G{}xWC{}'.format(i,i)])/calc_obs_MGCAMB['G{}xWC{}'.format(i,i)],
                        label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
       
    
    
    if 'WL' in use_obs and 'GWC' in use_obs:
        axes[j].set_title('WLxGW-NC',loc='right',fontdict={'fontsize':16})
        for i in range(1,Nbins_gc+1):
            axes[j].plot(ells,(calc_obs_MGCAMB['L{}xWC{}'.format(i,i)]-calc_obs_CAMB['L{}xWC{}'.format(i,i)])/calc_obs_MGCAMB['L{}xWC{}'.format(i,i)],
                        label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
       # axes[j].set_yscale('log')
        j+=1

    if 'GWWL' in use_obs and 'GWC' in use_obs:
        axes[j].set_title('GW-WLxGW-NC',loc='right',fontdict={'fontsize':16})
        for i in range(1,Nbins_gc+1):
            axes[j].plot(ells,(calc_obs_MGCAMB['WL{}xWC{}'.format(i,i)]-calc_obs_CAMB['WL{}xWC{}'.format(i,i)])/calc_obs_MGCAMB['WL{}xWC{}'.format(i,i)],
                        label=r'$i={}$'.format(i),color=bincolors[i-1])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        #axes[j].set_yscale('log')
        j+=1
    
    axes[0].set_ylabel(r'$\Delta C_{ii}^{AB}(\ell)[\%]$', fontsize=20)
   # plt.subplots_adjust(hspace=1.2)
    plt.savefig(plot_path+'_cross.pdf', bbox_inches='tight')
   



In [ ]:
norm = ells*(ells+1)/(2*np.pi)
j=0
color=['darkorchid', 'goldenrod', 'forestgreen']
bins = [1,7,10]
if 'GWWL' in use_obs or 'GWC'  in use_obs or 'GC' in use_obs:
    fig, axes = plt.subplots(ncols=len(use_obs), sharey=True, subplot_kw=dict(frameon=True),figsize=(12,4))
    plt.suptitle('Relative difference between $C_\ell$ evaluated with CAMB and MGCAMB, i=1,7,10',y=1.05)
    axis_label_fontsize = 20
    title_fontsize = 20
    title_fontweight = 'bold'
    if 'GC' in use_obs:
        axes[j].set_title('GCph',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['G{}xG{}'.format(k,k)]-calc_obs_CAMB['G{}xG{}'.format(k,k)])/calc_obs_MGCAMB['G{}xG{}'.format(k,k)],
                         label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1
        
    if 'WL' in use_obs:
        axes[j].set_title('WL',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['L{}xL{}'.format(k,k)]-calc_obs_CAMB['L{}xL{}'.format(k,k)])/calc_obs_MGCAMB['L{}xL{}'.format(k,k)],
                         label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        #axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1    
    
    if 'GWWL' in use_obs:
        axes[j].set_title('GW-WL',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['WL{}xWL{}'.format(k,k)]-calc_obs_CAMB['WL{}xWL{}'.format(k,k)])/calc_obs_MGCAMB['WL{}xWL{}'.format(k,k)],
                         label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        #axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1

    if 'GWC' in use_obs:
        axes[j].set_title('GW-NC',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['WC{}xWC{}'.format(k,k)]-calc_obs_CAMB['WC{}xWC{}'.format(k,k)])/calc_obs_MGCAMB['WC{}xWC{}'.format(k,k)],
                         label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        #axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1
        
    
    
    
    axes[-1].legend(**sidelegend)
    axes[0].set_ylabel(r'$\Delta C_{ii}^{AB}(\ell)[\%]$', fontsize=axis_label_fontsize)
    plt.subplots_adjust(hspace=1.2)
    plt.savefig(plot_path+'_firstbin.pdf', bbox_inches='tight')
    axes[-1].legend(**sidelegend);
    

In [ ]:
norm = ells*(ells+1)/(2*np.pi)
j=0
if 'GWWL' in use_obs or 'GWC'  in use_obs or 'GC' in use_obs:
    fig, axes = plt.subplots(ncols=len(use_obs)+2, sharey=False, subplot_kw=dict(frameon=True),figsize=(16,6))
    plt.suptitle('Relative difference between $C_\ell$ evaluated with CAMB and MGCAMB, i=1,7,10',y=1.05)
    axis_label_fontsize = 20
    title_fontsize = 20
    title_fontweight = 'bold'
    

    
    
    if 'GC' in use_obs and 'WL' in use_obs:
        axes[j].set_title('GGL',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['G{}xL{}'.format(k,k)]-calc_obs_CAMB['G{}xL{}'.format(k,k)])/calc_obs_MGCAMB['G{}xL{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
        
    if 'GC' in use_obs and 'GWWL' in use_obs:
        axes[j].set_title('GCphxGW-WL',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['G{}xWL{}'.format(k,k)]-calc_obs_CAMB['G{}xWL{}'.format(k,k)])/calc_obs_MGCAMB['G{}xWL{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
       
    
    
    if 'WL' in use_obs and 'GWWL' in use_obs:
        axes[j].set_title('WLxGW-WL',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['L{}xWL{}'.format(k,k)]-calc_obs_CAMB['L{}xWL{}'.format(k,k)])/calc_obs_MGCAMB['L{}xWL{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
        
    if 'GC' in use_obs and 'GWC' in use_obs:
        axes[j].set_title('GCphxGW-NC',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['G{}xWC{}'.format(k,k)]-calc_obs_CAMB['G{}xWC{}'.format(k,k)])/calc_obs_MGCAMB['G{}xWC{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
       
    
    
    if 'WL' in use_obs and 'GWC' in use_obs:
        axes[j].set_title('WLxGW-NC',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['L{}xWC{}'.format(k,k)]-calc_obs_CAMB['L{}xWC{}'.format(k,k)])/calc_obs_MGCAMB['L{}xWC{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
       # axes[j].set_yscale('log')
        j+=1

    if 'GWWL' in use_obs and 'GWC' in use_obs:
        axes[j].set_title('GW-WLxGW-NC',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['WL{}xWC{}'.format(k,k)]-calc_obs_CAMB['WL{}xWC{}'.format(k,k)])/calc_obs_MGCAMB['WL{}xWC{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        #axes[j].set_yscale('log')
        j+=1
    
    axes[0].set_ylabel(r'$\Delta C_{ii}^{AB}(\ell)[\%]$', fontsize=20)
   # plt.subplots_adjust(hspace=1.2)
    plt.savefig(plot_path+'_cross_firstbin.pdf', bbox_inches='tight')
   



## DIFFERENCE OF CLS DIVIDED BY SIGMA

In [ ]:

norm = ells*(ells+1)/(2*np.pi)
j=0
color=['darkorchid', 'goldenrod', 'forestgreen']
bins = [1,5,10]
if 'GWWL' in use_obs or 'GWC'  in use_obs or 'GC' in use_obs:
    fig, axes = plt.subplots(ncols=len(use_obs), sharey=True, subplot_kw=dict(frameon=True),figsize=(12,4))
    plt.suptitle('Relative difference between $C_\ell$ evaluated with CAMB and MGCAMB, i=1,5,10',y=1.05)
    axis_label_fontsize = 20
    title_fontsize = 20
    title_fontweight = 'bold'
    if 'GC' in use_obs:
        axes[j].set_title('GCph',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['G{}xG{}'.format(k,k)]-calc_obs_CAMB['G{}xG{}'.format(k,k)])/sigma['G{}xG{}'.format(k,k)],
                         label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1
        
    if 'WL' in use_obs:
        axes[j].set_title('WL',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['L{}xL{}'.format(k,k)]-calc_obs_CAMB['L{}xL{}'.format(k,k)])/sigma['L{}xL{}'.format(k,k)],
                         label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        #axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1    
    
    if 'GWWL' in use_obs:
        axes[j].set_title('GW-WL',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['WL{}xWL{}'.format(k,k)]-calc_obs_CAMB['WL{}xWL{}'.format(k,k)])/sigma['WL{}xWL{}'.format(k,k)],
                         label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        #axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1

    if 'GWC' in use_obs:
        axes[j].set_title('GW-NC',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,np.abs(calc_obs_MGCAMB['WC{}xWC{}'.format(k,k)]-calc_obs_CAMB['WC{}xWC{}'.format(k,k)])/sigma['WC{}xWC{}'.format(k,k)],
                         label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        #axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        axes[j].set_yscale('log')
        j+=1
        
    
    
    
    axes[-1].legend(**sidelegend)
    axes[0].set_ylabel(r'$\Delta C_{ii}^{AB}(\ell)/\sigma_{ii}^{AB}(\ell)$', fontsize=axis_label_fontsize)
    plt.subplots_adjust(hspace=1.2)
    plt.savefig(plot_path+'_firstbin.pdf', bbox_inches='tight')
    axes[-1].legend(**sidelegend);
    

In [ ]:
norm = ells*(ells+1)/(2*np.pi)
j=0
if 'GWWL' in use_obs or 'GWC'  in use_obs or 'GC' in use_obs:
    fig, axes = plt.subplots(ncols=len(use_obs)+2, sharey=False, subplot_kw=dict(frameon=True),figsize=(16,6))
    plt.suptitle('Relative difference between $C_\ell$ evaluated with CAMB and MGCAMB, i=1,5,10',y=1.05)
    axis_label_fontsize = 20
    title_fontsize = 20
    title_fontweight = 'bold'
    

    
    
    if 'GC' in use_obs and 'WL' in use_obs:
        axes[j].set_title('GGL',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['G{}xL{}'.format(k,k)]-calc_obs_CAMB['G{}xL{}'.format(k,k)])/sigma['G{}xL{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
        
    if 'GC' in use_obs and 'GWWL' in use_obs:
        axes[j].set_title('GCphxGW-WL',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['G{}xWL{}'.format(k,k)]-calc_obs_CAMB['G{}xWL{}'.format(k,k)])/sigma['G{}xWL{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
       
    
    
    if 'WL' in use_obs and 'GWWL' in use_obs:
        axes[j].set_title('WLxGW-WL',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['L{}xWL{}'.format(k,k)]-calc_obs_CAMB['L{}xWL{}'.format(k,k)])/sigma['L{}xWL{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
        
    if 'GC' in use_obs and 'GWC' in use_obs:
        axes[j].set_title('GCphxGW-NC',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['G{}xWC{}'.format(k,k)]-calc_obs_CAMB['G{}xWC{}'.format(k,k)])/sigma['G{}xWC{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        j+=1
       
    
    
    if 'WL' in use_obs and 'GWC' in use_obs:
        axes[j].set_title('WLxGW-NC',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['L{}xWC{}'.format(k,k)]-calc_obs_CAMB['L{}xWC{}'.format(k,k)])/sigma['L{}xWC{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
       # axes[j].set_yscale('log')
        j+=1

    if 'GWWL' in use_obs and 'GWC' in use_obs:
        axes[j].set_title('GW-WLxGW-NC',loc='right',fontdict={'fontsize':16})
        for i,k in enumerate(bins):
            axes[j].plot(ells,(calc_obs_MGCAMB['WL{}xWC{}'.format(k,k)]-calc_obs_CAMB['WL{}xWC{}'.format(k,k)])/sigma['WC{}xWL{}'.format(k,k)],
                        label=r'$i={}$'.format(k),color=color[i])
        axes[j].set_xlabel(r'$\ell$')
        axes[j].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
        axes[j].set_xscale('log')
        #axes[j].set_yscale('log')
        j+=1
    
    axes[0].set_ylabel(r'$\Delta C_{ii}^{AB}(\ell)/\sigma_{ii}^{AB}(\ell)$', fontsize=20)
   # plt.subplots_adjust(hspace=1.2)
    plt.savefig(plot_path+'_cross_firstbin.pdf', bbox_inches='tight')
    axes[-1].legend(**sidelegend);
   

